In [1]:
"""
======================================================
  Williams-Sonoma Event-Driven Demand Forecasting
  Dataset Generator (LSTM Augmented Scale - 1.1M Rows)
======================================================
"""

import pandas as pd
import numpy as np
from datetime import date, timedelta
import random
import os

# ─────────────────────────────────────────────
# SEED
# ─────────────────────────────────────────────
np.random.seed(42)
random.seed(42)

OUTPUT_DIR = "ws_demand_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# 1. PRODUCT HIERARCHY (WITH VARIANT AUGMENTATION)
# ─────────────────────────────────────────────
print("⚙️  Building augmented product hierarchy...")

# Base product templates
products = {
    "WS": [
        ("WS", "Williams-Sonoma", "Cookware",   "Cast Iron",     "WS-CI-001", "Le Creuset Dutch Oven 5.5qt",   30, "luxury"),
        ("WS", "Williams-Sonoma", "Cookware",   "Cast Iron",     "WS-CI-002", "Staub Cocotte 4qt",             25, "luxury"),
        ("WS", "Williams-Sonoma", "Cookware",   "Stainless",     "WS-SS-001", "All-Clad D3 10pc Set",          20, "luxury"),
        ("WS", "Williams-Sonoma", "Cookware",   "Stainless",     "WS-SS-002", "Demeyere Atlantis Skillet",     18, "luxury"),
        ("WS", "Williams-Sonoma", "Bakeware",   "Cake Pans",     "WS-BK-001", "Nordic Ware Bundt Pan",         45, "premium"),
        ("WS", "Williams-Sonoma", "Bakeware",   "Sheet Pans",    "WS-BK-002", "USA Pan Half Sheet",            60, "premium"),
        ("WS", "Williams-Sonoma", "Electrics",  "Stand Mixers",  "WS-EL-001", "KitchenAid Artisan 5qt",        35, "luxury"),
        ("WS", "Williams-Sonoma", "Electrics",  "Blenders",      "WS-EL-002", "Vitamix A3500",                 15, "luxury"),
        ("WS", "Williams-Sonoma", "Serveware",  "Platters",      "WS-SW-001", "Pillivuyt Oval Platter",        40, "premium"),
        ("WS", "Williams-Sonoma", "Serveware",  "Entertaining",  "WS-SW-002", "Juliska Berry Thread Bowl Set", 35, "premium"),
        ("WS", "Williams-Sonoma", "Cutlery",    "Knife Sets",    "WS-CU-001", "Wusthof Classic 7pc Block",     22, "luxury"),
        ("WS", "Williams-Sonoma", "Cutlery",    "Scissors",      "WS-CU-002", "Joyce Chen Unlimited Scissors", 55, "premium"),
    ],
    "PB": [
        ("PB", "Pottery Barn",    "Bedding",    "Duvet Covers",  "PB-BD-001", "Belgian Flax Linen Duvet",      28, "luxury"),
        ("PB", "Pottery Barn",    "Bedding",    "Sheet Sets",    "PB-BD-002", "Organic Sateen Sheet Set",      45, "premium"),
        ("PB", "Pottery Barn",    "Furniture",  "Sofas",         "PB-FU-001", "Comfort Square Arm Sofa",        5, "luxury"),
        ("PB", "Pottery Barn",    "Furniture",  "Accent Chairs", "PB-FU-002", "Remy Swivel Chair",              8, "luxury"),
        ("PB", "Pottery Barn",    "Decor",      "Throws",        "PB-DC-001", "Faux Fur Throw Blanket",        65, "premium"),
        ("PB", "Pottery Barn",    "Decor",      "Candles",       "PB-DC-002", "Volcano Candle Set",            80, "premium"),
        ("PB", "Pottery Barn",    "Dorm",       "Bedding",       "PB-DR-001", "Essential Dorm Bedding Bundle", 55, "premium"),
        ("PB", "Pottery Barn",    "Dorm",       "Storage",       "PB-DR-002", "Canvas Storage Bin Set",        70, "standard"),
        ("PB", "Pottery Barn",    "Lighting",   "Table Lamps",   "PB-LT-001", "Tilda Table Lamp",              20, "premium"),
        ("PB", "Pottery Barn",    "Lighting",   "Floor Lamps",   "PB-LT-002", "Callum Arc Floor Lamp",         12, "luxury"),
    ],
    "WE": [
        ("WE", "West Elm",        "Furniture",  "Sofas",         "WE-FU-001", "Andes Sofa",                     7, "premium"),
        ("WE", "West Elm",        "Furniture",  "Dining Tables", "WE-FU-002", "Emmerson Dining Table",          6, "premium"),
        ("WE", "West Elm",        "Decor",      "Pillows",       "WE-DC-001", "Velvet Lumbar Pillow Set",      75, "standard"),
        ("WE", "West Elm",        "Decor",      "Art",           "WE-DC-002", "Abstract Canvas Print",         30, "premium"),
        ("WE", "West Elm",        "Bedding",    "Duvet Covers",  "WE-BD-001", "Organic Cotton Percale Duvet",  35, "premium"),
        ("WE", "West Elm",        "Bedding",    "Quilts",        "WE-BD-002", "Stripe Organic Cotton Quilt",   28, "premium"),
        ("WE", "West Elm",        "Rugs",       "Area Rugs",     "WE-RG-001", "Jute Boucle Rug 8x10",          15, "premium"),
        ("WE", "West Elm",        "Rugs",       "Runners",       "WE-RG-002", "Carved Geo Runner",             22, "standard"),
    ],
}

# Scale multiplier: Generate 20 variants for every base SKU
VARIANTS_PER_BASE = 20
variant_attributes = ["Matte Black", "Cherry", "White", "Navy", "Brass", "Chrome", "Linen",
                      "Charcoal", "Blush", "Sage", "Set of 2", "Set of 4", "Oversized",
                      "Petite", "Classic", "Modern", "Eco-Friendly", "Limited Edition", "Signature", "Essential"]

all_products = []
for brand_id, brand_skus in products.items():
    for p in brand_skus:
        for v_idx in range(VARIANTS_PER_BASE):
            # Give each variant a unique ID and name
            variant_sku = f"{p[4]}-V{v_idx:02d}"
            variant_name = f"{p[5]} - {variant_attributes[v_idx]}"

            # Inject noise so variants have different popularity baselines
            demand_variance = np.random.uniform(0.3, 1.7)
            variant_base_demand = max(2, int(p[6] * demand_variance))

            all_products.append({
                "brand_id":       p[0],
                "brand_name":     p[1],
                "category":       p[2],
                "subcategory":    p[3],
                "sku_id":         variant_sku,
                "sku_name":       variant_name,
                "base_demand":    variant_base_demand,
                "price_tier":     p[7],
            })

product_df = pd.DataFrame(all_products)
product_df.to_csv(f"{OUTPUT_DIR}/product_hierarchy.csv", index=False)
print(f"   ✅ Scaled to {len(product_df)} unique SKUs")

# ─────────────────────────────────────────────
# 2. EVENTS DATA (EXPANDED TO 5 YEARS)
# ─────────────────────────────────────────────
print("⚙️  Building expanded events calendar...")

def get_thanksgiving(year):
    nov1 = date(year, 11, 1)
    first_thursday = nov1 + timedelta(days=(3 - nov1.weekday()) % 7)
    return first_thursday + timedelta(weeks=3)

def get_mothers_day(year):
    may1 = date(year, 5, 1)
    first_sunday = may1 + timedelta(days=(6 - may1.weekday()) % 7)
    return first_sunday + timedelta(weeks=1)

def get_superbowl(year):
    feb1 = date(year, 2, 1)
    first_sunday = feb1 + timedelta(days=(6 - feb1.weekday()) % 7)
    return first_sunday + timedelta(weeks=1)

# Extended timeline for Deep Learning
YEARS = [2020, 2021, 2022, 2023, 2024]

event_rows = []
for yr in YEARS:
    tg = get_thanksgiving(yr)
    events_this_year = [
        ("Thanksgiving",     tg,                  28, 3,  "WS",       "Cookware,Bakeware,Serveware,Cutlery"),
        ("Black Friday",     tg + timedelta(1),  21, 5,  "ALL",      "ALL"),
        ("Cyber Monday",     tg + timedelta(4),  14, 3,  "ALL",      "ALL"),
        ("Christmas",        date(yr, 12, 25),   35, 10, "ALL",      "ALL"),
        ("New Year",         date(yr+1, 1, 1),   14, 5,  "ALL",      "Cookware,Decor,Bedding"),
        ("Valentines Day",   date(yr, 2, 14),    14, 3,  "WS,PB",    "Cookware,Decor,Lighting"),
        ("Mothers Day",      get_mothers_day(yr), 21, 3, "WS,PB",    "Cookware,Bakeware,Serveware"),
        ("Super Bowl",       get_superbowl(yr),  14, 2,  "WS",       "Cookware,Serveware,Electrics"),
        ("Wedding Season",   date(yr, 5, 15),    45, 14, "WS,PB",    "ALL"),
        ("Back to College",  date(yr, 8, 1),     35, 7,  "PB",       "Dorm,Bedding,Lighting"),
        ("Labor Day",        date(yr, 9, 7),     14, 3,  "WE,PB",    "Furniture,Rugs"),
    ]
    for ev in events_this_year:
        event_rows.append({
            "event_name":          ev[0],
            "event_date":          ev[1],
            "pre_impact_window":   ev[2],
            "post_impact_window":  ev[3],
            "affected_brands":     ev[4],
            "affected_categories": ev[5],
        })

events_df = pd.DataFrame(event_rows)
events_df.to_csv(f"{OUTPUT_DIR}/events_data.csv", index=False)
print(f"   ✅ {len(events_df)} event records across {len(YEARS)} years")

# ─────────────────────────────────────────────
# 3. CUSTOMER SEGMENTS (UNCHANGED)
# ─────────────────────────────────────────────
print("⚙️  Building customer segments...")
segments = [
    ("SEG-01", "Luxury Self-Buyer",    "high",   "online",  False, False, 0.90),
    ("SEG-02", "Gift Buyer Premium",   "high",   "online",  True,  False, 0.85),
    ("SEG-03", "Registry Buyer",       "high",   "online",  True,  True,  0.95),
    ("SEG-04", "In-Store Loyalist",    "high",   "store",   False, False, 0.88),
    ("SEG-05", "Catalog Buyer",        "medium", "catalog", False, False, 0.75),
    ("SEG-06", "Aspirational Buyer",   "medium", "online",  False, False, 0.70),
    ("SEG-07", "Gift Buyer Standard",  "medium", "online",  True,  False, 0.72),
    ("SEG-08", "Dorm/College Buyer",   "low",    "online",  False, False, 0.65),
    ("SEG-09", "New Customer",         "low",    "online",  False, False, 0.60),
    ("SEG-10", "Deal Seeker",          "low",    "online",  False, False, 0.55),
]
seg_df = pd.DataFrame(segments, columns=["segment_id", "segment_name", "ltv_tier", "channel", "is_gift_buyer", "is_registry_buyer", "conversion_rate"])
seg_df.to_csv(f"{OUTPUT_DIR}/customer_segments.csv", index=False)
print(f"   ✅ {len(seg_df)} customer segments")

# ─────────────────────────────────────────────
# 4. REGISTRY DATA (EXPANDED DATES)
# ─────────────────────────────────────────────
print("⚙️  Building registry signals...")

registry_skus = [p["sku_id"] for p in all_products if p["brand_id"] in ("WS", "PB")]
registry_rows = []
date_range = pd.date_range("2020-01-01", "2024-12-31", freq="MS") # Extended to 5 years

for sku in registry_skus:
    base = np.random.randint(5, 40)
    for dt in date_range:
        wedding_boost = 1.8 if dt.month in (5, 6, 7) else 1.0
        holiday_boost = 1.5 if dt.month in (11, 12) else 1.0
        noise = np.random.normal(1.0, 0.1)
        registry_rows.append({
            "sku_id":                      sku,
            "year_month":                  dt.strftime("%Y-%m"),
            "active_registries":           int(base * wedding_boost * holiday_boost * noise),
            "items_fulfilled_this_month":  int(base * 0.6 * wedding_boost * holiday_boost * noise),
        })

registry_df = pd.DataFrame(registry_rows)
registry_df.to_csv(f"{OUTPUT_DIR}/registry_data.csv", index=False)
print(f"   ✅ {len(registry_df)} registry records")

# ─────────────────────────────────────────────
# 5. SALES DATA (MASSIVE EXPANSION)
# ─────────────────────────────────────────────
print("⚙️  Generating massive sales data array (this will take a minute or two)...")

START_DATE = date(2020, 1, 1)
END_DATE   = date(2024, 12, 31)
all_dates  = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)]

def compute_event_effect(current_date, events_df, sku_brand, sku_category):
    total_effect = 1.0
    for _, ev in events_df.iterrows():
        brands_affected = ev["affected_brands"]
        cats_affected   = ev["affected_categories"]
        if brands_affected != "ALL" and sku_brand not in brands_affected: continue
        if cats_affected != "ALL" and sku_category not in cats_affected: continue

        ev_date  = pd.Timestamp(ev["event_date"]).date()
        pre_win  = ev["pre_impact_window"]
        post_win = ev["post_impact_window"]
        days_delta = (ev_date - current_date).days

        if 0 <= days_delta <= pre_win:
            tau    = max(pre_win / 4, 1.0)
            effect = np.exp(-days_delta / tau)
            total_effect += effect * 1.5
        elif -post_win <= days_delta < 0:
            days_after = abs(days_delta)
            tau_post   = max(post_win / 2, 1.0)
            effect     = np.exp(-days_after / tau_post)
            total_effect -= effect * 0.3

    return max(total_effect, 0.1)

def get_seasonality(dt, brand_id, category):
    m = dt.month
    q4_mult = {10: 1.3, 11: 1.8, 12: 2.5}.get(m, 1.0)
    dorm_mult = 1.5 if (brand_id == "PB" and category == "Dorm" and m in (7, 8)) else 1.0
    spring_mult = 1.2 if (category in ("Furniture", "Rugs") and m in (3, 4, 5)) else 1.0
    return q4_mult * dorm_mult * spring_mult

def get_dow_multiplier(dt):
    return {0: 1.0, 1: 0.9, 2: 0.9, 3: 0.95, 4: 1.1, 5: 1.3, 6: 1.2}.get(dt.weekday(), 1.0)

def get_yoy_growth(dt):
    # Updated to start at 2020 baseline
    return 1.0 + (dt.year - 2020) * 0.07

sales_rows = []
segments_list = seg_df["segment_id"].tolist()
seg_weights   = seg_df["conversion_rate"].tolist()

# Pre-calculate event effects to speed up the massive loop
event_cache = {}

for product in all_products:
    sku_id    = product["sku_id"]
    brand_id  = product["brand_id"]
    category  = product["category"]
    base_d    = product["base_demand"]

    # Cache key for events (brand + category mapping)
    cache_key = f"{brand_id}_{category}"
    if cache_key not in event_cache:
        event_cache[cache_key] = [compute_event_effect(dt, events_df, brand_id, category) for dt in all_dates]

    for idx, dt in enumerate(all_dates):
        seasonality   = get_seasonality(dt, brand_id, category)
        event_effect  = event_cache[cache_key][idx]
        dow_mult      = get_dow_multiplier(dt)
        yoy_growth    = get_yoy_growth(dt)
        noise         = max(np.random.normal(1.0, 0.12), 0.5) # Slightly higher noise for DL generalization

        expected_units = (base_d * seasonality * event_effect * dow_mult * yoy_growth * noise)
        units_sold = max(int(round(expected_units)), 0)

        seg_id = random.choices(segments_list, weights=seg_weights, k=1)[0]

        sales_rows.append({
            "date":             dt.isoformat(),
            "sku_id":           sku_id,
            "brand_id":         brand_id,
            "category":         category,
            "segment_id":       seg_id,
            "units_sold":       units_sold,
            "seasonality_mult": round(seasonality, 3),
            "event_effect":     round(event_effect, 3),
        })

sales_df = pd.DataFrame(sales_rows)
sales_df.to_csv(f"{OUTPUT_DIR}/sales_data.csv", index=False)
print(f"   ✅ {len(sales_df):,} total sales records generated.")
print(f"       Date range : {sales_df['date'].min()} → {sales_df['date'].max()}")
print(f"       SKUs       : {sales_df['sku_id'].nunique()}")
print("✅ Done! All files saved.")

⚙️  Building augmented product hierarchy...
   ✅ Scaled to 600 unique SKUs
⚙️  Building expanded events calendar...
   ✅ 55 event records across 5 years
⚙️  Building customer segments...
   ✅ 10 customer segments
⚙️  Building registry signals...
   ✅ 26400 registry records
⚙️  Generating massive sales data array (this will take a minute or two)...
   ✅ 1,096,200 total sales records generated.
       Date range : 2020-01-01 → 2024-12-31
       SKUs       : 600
✅ Done! All files saved.


In [2]:
"""
======================================================
  Williams-Sonoma Demand Forecasting
  Step 2: LSTM Tensor Preparation (Sequence Windowing)
======================================================
  LSTMs require data in 3D format: [samples, time_steps, features]
  This script:
    1. Sorts data chronologically per SKU
    2. Scales continuous features (Crucial for Neural Networks)
    3. Generates 30-day sliding windows
    4. Saves arrays as .npy files for fast loading
======================================================
"""

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

DATA_DIR = "ws_demand_dataset"
SEQ_LENGTH = 30  # Look back 30 days to predict the next day
SPLIT_DATE = "2024-10-01" # Keep consistent with your XGBoost split

print("=" * 55)
print("  WS Forecasting — LSTM Tensor Preparation")
print("=" * 55)

# 1. Load Data
print("📂 Loading massive sales dataset...")
df = pd.read_csv(f"{DATA_DIR}/sales_data.csv", parse_dates=["date"])
df = df.sort_values(by=["sku_id", "date"]).reset_index(drop=True)

# 2. Extract Basic Time Features
print("⚙️ Engineering time & categorical features...")
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# One-hot encode categorical variables
df = pd.get_dummies(df, columns=["brand_id", "category", "segment_id"], drop_first=True)

# 3. Define Features and Target
TARGET_COL = "units_sold"
META_COLS = ["date", "sku_id"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS + [TARGET_COL]]

# 4. Train/Test Split (Time-based)
print(f"✂️ Splitting data at {SPLIT_DATE}...")
train_df = df[df["date"] < SPLIT_DATE].copy()
test_df = df[df["date"] >= SPLIT_DATE].copy()

# 5. Scaling (Fit ONLY on train data to prevent data leakage)
print("📏 Scaling continuous features (MinMaxScaler)...")
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler() # Scale target separately so we can inverse_transform later

train_df[FEATURE_COLS] = feature_scaler.fit_transform(train_df[FEATURE_COLS])
train_df[[TARGET_COL]] = target_scaler.fit_transform(train_df[[TARGET_COL]])

test_df[FEATURE_COLS] = feature_scaler.transform(test_df[FEATURE_COLS])
test_df[[TARGET_COL]] = target_scaler.transform(test_df[[TARGET_COL]])

# 6. Create 3D Sequences [samples, timesteps, features]
def create_sequences(data, feature_cols, target_col, seq_length):
    X, y = [], []
    grouped = data.groupby("sku_id")

    for sku, group in grouped:
        group_features = group[feature_cols].values
        group_target = group[target_col].values

        # Sliding window
        for i in range(len(group) - seq_length):
            X.append(group_features[i : i + seq_length])
            y.append(group_target[i + seq_length])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

print(f"🪟 Creating {SEQ_LENGTH}-day sliding windows (This will take a moment for 1.1M rows)...")

X_train, y_train = create_sequences(train_df, FEATURE_COLS, TARGET_COL, SEQ_LENGTH)
X_test, y_test = create_sequences(test_df, FEATURE_COLS, TARGET_COL, SEQ_LENGTH)

print(f"\n✅ Tensor Shapes:")
print(f"   X_train : {X_train.shape}  [samples, timesteps, features]")
print(f"   y_train : {y_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_test  : {y_test.shape}")

# 7. Save Tensors and Scaler
print("\n💾 Saving tensors to disk...")
np.save(f"{DATA_DIR}/X_train.npy", X_train)
np.save(f"{DATA_DIR}/y_train.npy", y_train)
np.save(f"{DATA_DIR}/X_test.npy", X_test)
np.save(f"{DATA_DIR}/y_test.npy", y_test)

# Save the target scaler so we can decode predictions back to actual unit volumes
import joblib
joblib.dump(target_scaler, f"{DATA_DIR}/target_scaler.pkl")

print("✅ Step 2 Complete. Ready for LSTM Training.")


  WS Forecasting — LSTM Tensor Preparation
📂 Loading massive sales dataset...
⚙️ Engineering time & categorical features...
✂️ Splitting data at 2024-10-01...
📏 Scaling continuous features (MinMaxScaler)...
🪟 Creating 30-day sliding windows (This will take a moment for 1.1M rows)...

✅ Tensor Shapes:
   X_train : (1023000, 30, 26)  [samples, timesteps, features]
   y_train : (1023000,)
   X_test  : (37200, 30, 26)
   y_test  : (37200,)

💾 Saving tensors to disk...
✅ Step 2 Complete. Ready for LSTM Training.


In [3]:
"""
======================================================
  Williams-Sonoma Demand Forecasting
  Step 3: LSTM Neural Network Training
======================================================
"""

import numpy as np
import tensorflow as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os

DATA_DIR = "ws_demand_dataset"
MODEL_DIR = "ws_model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("=" * 55)
print("  WS Forecasting — LSTM Training")
print("=" * 55)

# 1. Load Data
print("📂 Loading Tensors...")
X_train = np.load(f"{DATA_DIR}/X_train.npy")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
X_test = np.load(f"{DATA_DIR}/X_test.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
target_scaler = joblib.load(f"{DATA_DIR}/target_scaler.pkl")

timesteps = X_train.shape[1]
features = X_train.shape[2]

# 2. Build LSTM Architecture
print(f"🧠 Building LSTM Architecture (Input Shape: {timesteps} steps, {features} features)...")

model = Sequential([
    Input(shape=(timesteps, features)),

    # Layer 1: LSTM returns sequences so the next LSTM can read them
    LSTM(64, return_sequences=True, activation='tanh'),
    Dropout(0.2), # Prevent overfitting

    # Layer 2: Final LSTM condenses time dimension
    LSTM(32, activation='tanh'),
    Dropout(0.2),

    # Output Layer: Predicts next day's scaled demand
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# 3. Training Callbacks
# Stop early if validation loss doesn't improve for 5 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
# Reduce learning rate if plateaued to fine-tune weights
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)

# 4. Train Model
print("\n🚀 Commencing Training (Batch Size 256 for large dataset)...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# 5. Predict & Evaluate
print("\n📈 Generating Predictions on Test Set (Holiday Season 2024)...")
preds_scaled = model.predict(X_test)

# Inverse transform to get actual unit quantities
y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
preds_actual = target_scaler.inverse_transform(preds_scaled).flatten()

# Floor predictions at 0 (can't sell negative items)
preds_actual = np.maximum(preds_actual, 0)

# Custom WMAPE metric
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

wm = wmape(y_test_actual, preds_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
mae = mean_absolute_error(y_test_actual, preds_actual)

print("\n" + "─"*45)
print("  Deep Learning Results (Test Set)")
print("─"*45)
print(f"  WMAPE  : {wm:6.2f}%")
print(f"  RMSE   : {rmse:7.2f} units")
print(f"  MAE    : {mae:7.2f} units")
print("─"*45)

# 6. Save Model
print("\n💾 Saving LSTM model...")
model.save(f"{MODEL_DIR}/ws_lstm_forecaster.keras")
print("✅ Complete! Pipeline successfully executed.")

  WS Forecasting — LSTM Training
📂 Loading Tensors...
🧠 Building LSTM Architecture (Input Shape: 30 steps, 26 features)...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        23,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,257 (141.63 KB)

 Trainable params: 36,257 (141.63 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Commencing Training (Batch Size 256 for large dataset)...
Epoch 1/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 425s 124ms/step - loss: 0.0013 - mae: 0.0209 - val_loss: 5.5066e-04 - val_mae: 0.0158 - learning_rate: 0.0010
Epoch 2/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 433s 127ms/step - loss: 0.0011 - mae: 0.0197 - val_loss: 5.5183e-04 - val_mae: 0.0164 - learning_rate: 0.0010
Epoch 3/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 413s 122ms/step - loss: 0.0011 - mae: 0.0195 - val_loss: 5.7643e-04 - val_mae: 0.0169 - learning_rate: 0.0010
Epoch 4/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 420s 124ms/step - loss: 0.0010 - mae: 0.0193 - val_loss: 6.1915e-04 - val_mae: 0.0182 - learning_rate: 5.0000e-04
Epoch 5/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 419s 123ms/step - loss: 0.0010 - mae: 0.0192 - val_loss: 6.8430e-04 - val_mae: 0.0198 - learning_rate: 5.0000e-04
Epoch 6/30
3397/3397 ━━━━━━━━━━━━━━━━━━━━ 411s 121ms/step - loss: 0.0010 - mae: 0.0190 - val_loss: 6.7264e-04 - val_mae: 0.0193 - learning_rate: 2.5000e-04

📈 Generating P